# Applying SAM 3 to map text PNGs

In [1]:
# Imports
from pathlib import Path
from zipfile import ZipFile
from PIL import Image
from pandas import read_csv as pandas_read_csv
from geopandas import read_file as read_geo_file
from transformers import Sam3Processor, Sam3Model

In [11]:
# Assessing data files inside zip file
with ZipFile(r"./Os-Historic-Batch1.zip") as zip:
    print(*(x for x in zip.namelist() if x.endswith(".gpkg")), sep = ",\n")

pngs/control-points.gpkg,
pngs/text-locations.gpkg


In [12]:
# Check this matches one of the file names printed out above
point_prompts_fp = "pngs/text-locations.gpkg"

In [15]:
# Load point prompts data
with ZipFile(r"./Os-Historic-Batch1.zip", "r") as zip:
    with zip.open(r"pngs/text-locations.gpkg", "r") as temp:
        point_prompts_meta = read_geo_file(temp)

point_prompts_meta.head()

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: File /vsimem/pyogrio_e1b98ba14350426f8bd86fa6fb35d06e has GPKG application_id, but non conformant file extension
  return ogr_read(


,pin_id,final_text,nation,osgb_east,osgb_north,tiff_filename,png_filename,pixel_x,pixel_y,geometry
0,5273fd1b6fe6180371000bf4,Lodge,Wales,311794.956153,291870.508674,mont-so19sw-2.tif,mont-so19sw-2-7.png,2154,3755,POINT (311794.956 291870.509)
1,527401136fe6180371000d91,St. Giles's Cottage,Wales,312234.410646,292092.145767,mont-so19sw-2.tif,mont-so19sw-2-7.png,2681,3489,POINT (312234.411 292092.146)
2,527401136fe6180371000d91,St. Giles's Cottage,Wales,312234.410646,292092.145767,mont-so19sw-2.tif,mont-so19sw-2-6.png,2681,3489,POINT (312234.411 292092.146)
3,5274018e6fe6180714000067,Port House,Wales,312912.551496,292634.944871,mont-so19sw-2.tif,mont-so19sw-2-6.png,3494,2837,POINT (312912.551 292634.945)
4,5274018e6fe6180714000067,Port House,Wales,312912.551496,292634.944871,mont-so19sw-2.tif,mont-so19sw-2-10.png,3494,2837,POINT (312912.551 292634.945)


## Extracting point prompts

`point_prompts` should be a nested list of dimensions $(N, M_n, 1, 2)$ where:
- $N$ equals the number of pngs
- $M_n$ equals the number of labelled text instances within image $n$.
- $1$ represents that each text instance in the GB1900 gazetteer is only labelled with a single point.
- $2$ represents the $x,y$ coordinates for prompt point.

`point_labels` is a nested list of dimensions $(N, M_n, 1)$, that indicates whether the point prompt is "positive" $(1)$ or "negative" $(0)$. **NOTE** all points are positive.

In [29]:
image_filenames = sorted(point_prompts_meta['png_filename'].unique())
point_prompts = []
point_labels = []
for png in image_filenames:
    temp = point_prompts_meta.loc[
        (point_prompts_meta['png_filename'] == png), ["pixel_x", "pixel_y"]
    ]
    point_prompts.append(temp.values[:, None, :].tolist())
    point_labels.append([[1]] * len(temp))

## Extracting PNGs

[[1], [1], [1], [1], [1], [1], [1], [1], [1], [1]]